# ChatUniTest LoRA Fine-tuning

**Goal**: Fine-tune CodeLlama-7b-Instruct with QLoRA to generate Java JUnit tests

**Estimated time**: 3 hours (L4)

**Estimated cost**: ~$1.5-2

**Before running**:
- Runtime → Change runtime type → **L4 GPU**
- Make sure your HuggingFace token (write access) is ready

## Step 1: Install Dependencies

In [2]:
!pip install -q --force-reinstall --no-cache-dir bitsandbytes
!pip install -q transformers==4.44.0 peft==0.11.1 trl==0.9.6 \
    accelerate==0.33.0 datasets sentencepiece huggingface_hub

# Verify GPU and bitsandbytes
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import bitsandbytes
print(f"bitsandbytes version: {bitsandbytes.__version__}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 2.1.4 requires numpy<2,>=1.26.0; python_version >= "3.12", but you have numpy 2.4.4 which is incompatible.
datasets 4.8.4 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
matplotlib 3.8.2 requires numpy<2,>=1.21, but you have numpy 2.4.4 which is incompatible.
accelerate 0.33.0 requires numpy<2.0.0,>=1.17, but you have numpy 2.4.4 which is incompatible.
torchvision 0.23.0+cu128 requires torch==2.8.0, but you have torch 2.11.0 which is incompatible.
scipy 1.11.4 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.4.4 which is incompatible.
trl 0.9.6 requires numpy<2.0.0,>=1.18.2, but you have numpy 2.4.4 which is incompatible.
scikit-learn 1.3.2 requires numpy<2.0,>=1.17.3, but you have numpy 2.4.4 which is incompatible.
ERROR: pip's dependency resolver does no

## Step 2: Login to HuggingFace (write token required)

In [4]:
from huggingface_hub import login, notebook_login

# Option 1: Interactive login (recommended)
notebook_login()

# Option 2: Direct token (not recommended for shared notebooks)
# login(token="hf_xxxxxxxxxxxxx")

Token has not been saved to git credential helper.


## Step 3: Prepare Dataset

In [4]:
from datasets import load_dataset, Dataset
import pandas as pd
import re

# ── Prompt template (must match model_server.py inference format) ──
PROMPT_TEMPLATE = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
{context}
runtimeFacts:

### JUnit Test:
"""

def build_full_text(context: str, test: str) -> str:
    return PROMPT_TEMPLATE.format(context=context.strip()) + test.strip()

def has_assertion(test: str) -> bool:
    return any(kw in test for kw in ["assert", "Assert", "verify", "Verify", "fail("])

def is_valid_java(code: str) -> bool:
    return code.count("{") > 0 and abs(code.count("{") - code.count("}")) <= 2

def estimate_tokens(text: str) -> int:
    return len(text) // 4

# Load dataset
print("Loading dataset...")
raw = load_dataset("zzzghttt/context2test", split="train")
print(f"Raw samples: {len(raw)}")
print(f"Columns: {raw.column_names}")

# Auto-detect column names
col_context = next((c for c in ["context", "input", "source"] if c in raw.column_names), None)
col_test    = next((t for t in ["test", "output", "target"] if t in raw.column_names), None)
print(f"Using: context='{col_context}', test='{col_test}'")

df = raw.to_pandas()[[col_context, col_test]].copy()
df.columns = ["context", "test"]

# ── Data cleaning ──
before = len(df)
df = df.drop_duplicates(subset=["context"])
print(f"After dedup: {len(df)} (removed {before - len(df)})")

df = df[df["test"].apply(has_assertion)]
print(f"After assertion filter: {len(df)}")

df = df[df["test"].apply(is_valid_java)]
print(f"After Java structure filter: {len(df)}")

df = df[df["context"].str.strip().str.len() > 30]
print(f"After empty context filter: {len(df)}")

# Build full training text
df["text"] = df.apply(lambda r: build_full_text(r["context"], r["test"]), axis=1)
df["token_est"] = df["text"].apply(estimate_tokens)
df = df[df["token_est"] <= 2048]
print(f"After token length filter: {len(df)}")

# Sample top 5000
df = df.sample(frac=1, random_state=42).reset_index(drop=True).head(5000)
print(f"\nFinal training samples: {len(df)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(df[["text"]])

# Preview first sample
print("\n=== Sample preview (first 500 chars) ===")
print(train_dataset[0]["text"][:500])

Loading dataset...
Raw samples: 534827
Columns: ['input', 'context', 'output']
Using: context='context', test='output'
After dedup: 279722 (removed 255105)
After assertion filter: 248057
After Java structure filter: 248028
After empty context filter: 248028
After token length filter: 247656

Final training samples: 5000

=== Sample preview (first 500 chars) ===
mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
AbstractLink implements Link<L> { @Override public Supplier<InterledgerAddress> getOperatorAddressSupplier() { return operatorAddressSupplier; } protected  AbstractLink(
      final Supplier<InterledgerAddress> operatorAddressSupplier,
      final L linkSettings
  ); @Override LinkId getLinkId(); @Override void setLinkId(final LinkId linkId); @Override Supplier<InterledgerAddress> getOperatorAddressSupplier(); @Override L g


## Step 4: Load Base Model (QLoRA 4-bit)

In [5]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

BASE_MODEL = "codellama/CodeLlama-7b-Instruct-hf"

# 4-bit QLoRA configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. VRAM usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading tokenizer...


Loading model (4-bit)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded. VRAM usage: 3.92 GB


## Step 5: Configure LoRA

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,                    # Original TestGen2-lora uses r=64; r=32 reduces training time
    lora_alpha=64,           # alpha = 2*r for better stability
    target_modules=[
        "q_proj", "v_proj",  # Original config
        "k_proj", "o_proj",  # Extended: full attention coverage
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params ~8M / 7B total (~0.1%)

trainable params: 33,554,432 || all params: 6,772,101,120 || trainable%: 0.4955


## Step 6: Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# ── Set your HuggingFace username here ──
HF_USERNAME = "Leon-20292783"   # <-- replace with yours
OUTPUT_MODEL = f"{HF_USERNAME}/my-testgen-lora"

training_args = TrainingArguments(
    output_dir="./my-testgen-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,       # effective batch_size = 32
    gradient_checkpointing=True,         # save VRAM
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,                           # use fp16 (compatible with both L4 and A100)
    bf16=False,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,                      # checkpoint every 100 steps (resume on disconnect)
    save_total_limit=3,
    report_to="none",                    # disable wandb
    optim="paged_adamw_32bit",           # recommended optimizer for QLoRA
    max_grad_norm=0.3,
    group_by_length=True,                # group similar-length samples to reduce padding
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,                 # must match inference cutoff_len
    packing=False,
)

print("Starting training...")
print(f"Total steps: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
trainer.train()

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Starting training...
Total steps: 156


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,0.894100
20,0.791200
30,0.823100
40,0.788200
50,0.673300
60,0.710800
70,0.753600
80,0.716700
90,0.670300
100,0.700900


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=156, training_loss=0.737511647053254, metrics={'train_runtime': 4843.0805, 'train_samples_per_second': 1.032, 'train_steps_per_second': 0.032, 'total_flos': 9.924420675826483e+16, 'train_loss': 0.737511647053254, 'epoch': 0.9984})

In [9]:
  HF_USERNAME = "Leon-20292783"   # 改成你的真实用户名
  OUTPUT_MODEL = f"{HF_USERNAME}/my-testgen-lora"

## Step 7: Save and Push to HuggingFace Hub

In [10]:
print("Saving model...")
trainer.save_model("./my-testgen-lora")

print(f"Pushing to HuggingFace Hub: {OUTPUT_MODEL}")
model.push_to_hub(OUTPUT_MODEL, private=False)
tokenizer.push_to_hub(OUTPUT_MODEL, private=False)

print(f"\nDone! Model available at: https://huggingface.co/{OUTPUT_MODEL}")
print(f"\nNext step — update model_server.py line 23:")
print(f'  PeftModel.from_pretrained(model, "{OUTPUT_MODEL}")')

Saving model...
Pushing to HuggingFace Hub: Leon-20292783/my-testgen-lora


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Done! Model available at: https://huggingface.co/Leon-20292783/my-testgen-lora

Next step — update model_server.py line 23:
  PeftModel.from_pretrained(model, "Leon-20292783/my-testgen-lora")


## Step 8 (Optional): Resume from Checkpoint

If Colab disconnects, use this cell to resume from the latest checkpoint.

In [1]:
import os

# Find latest checkpoint
checkpoints = [
    d for d in os.listdir("./my-testgen-lora")
    if d.startswith("checkpoint-")
]
if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    resume_from = f"./my-testgen-lora/{latest}"
    print(f"Resuming from {resume_from}")
    trainer.train(resume_from_checkpoint=resume_from)
else:
    print("No checkpoint found — please run from Step 1")

Resuming from ./my-testgen-lora/checkpoint-156


NameError: name 'trainer' is not defined

## Step 9-Pre: Recover Model Variables

In [3]:
  import torch
  from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
  from peft import PeftModel

  HF_USERNAME = "Leon-20292783"   # 你的用户名
  OUTPUT_MODEL = f"{HF_USERNAME}/my-testgen-lora"
  BASE_MODEL = "codellama/CodeLlama-7b-Instruct-hf"

  bnb_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_use_double_quant=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.bfloat16,
  )

  tokenizer = AutoTokenizer.from_pretrained(OUTPUT_MODEL)

  model = AutoModelForCausalLM.from_pretrained(
      BASE_MODEL,
      quantization_config=bnb_config,
      device_map="auto",
      torch_dtype=torch.bfloat16,
  )
  model = PeftModel.from_pretrained(model, OUTPUT_MODEL)
  model.eval()
  print("Model loaded.")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: bc3e342b-7328-4109-8b28-6b87bcf89034)')' thrown while requesting HEAD https://huggingface.co/Leon-20292783/my-testgen-lora/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 2b8f4581-14b5-49ed-9765-d5a52925eb15)')' thrown while requesting HEAD https://huggingface.co/Leon-20292783/my-testgen-lora/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].


KeyboardInterrupt: 

## Step 9 (Optional): Quick Generation Test

In [8]:
from transformers import GenerationConfig

# Test with the same prompt format used in model_server.py
test_prompt = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
public class PDFTextStripper {
    public String getText(PDDocument doc) throws IOException {
        StringWriter writer = new StringWriter();
        writeText(doc, writer);
        return writer.toString();
    }
}
runtimeFacts:

### JUnit Test:
"""

model.eval()
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.6,
        do_sample=True,
        top_p=0.95,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== Generated output ===")
print(generated)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== Generated output ===
@Test public void testGetText() throws Exception { assertNotNull("PDFTextStripper.getText should not be null", stripper.getText(document)); }  @Test public void testGetText() throws Exception; }
runtimeFacts:

### JUnit Test:
@Test public void testGetText() throws Exception { assertNotNull("PDFTextStripper.getText should not be null", stripper.getText(document)); }  @Test public void testGetText() throws Exception; }
runtimeFacts:

### JUnit Test:
@Test public void testGetText() throws Exception { assertNotNull("PDFTextStripper.getText should not be null", stripper.getText(document)); }  @Test public void testGetText() throws Exception; }
runtimeFacts:

### JUnit Test:
@Test public void testGetText() throws Exception { assertNotNull("PDFTextStripper.getText should not be null", stripper.getText(document)); }  @Test public void testGetText() throws Exception; }
runtimeFacts:

### JUnit Test:
@Test public void testGetText()
